In [ ]:
from pathlib import Path
import ast

import pandas as pd

In [ ]:
fp = Path("DST/extracted_entities.csv")


# fp = Path("data/extracted_entities_2026_06_03.csv")
df_input = pd.read_csv(fp)
df_input = df_input.rename(columns={"geographic location" : "location"})

#, index_col=0)
# Entity values to python objects
entity_cols = ['person', 'location', 'organization']
df_input[entity_cols] = df_input[entity_cols].map(ast.literal_eval)
df_input

In [ ]:
# Optional: reduce irrelevant names by only keeping sentences that contain 'Nias'
# and the sentences before and after it (if they are from the same article)

df = df_input.copy()

# If you don't want to filter, comment out the following lines:

df['relevant_sentence'] = False
df['relevant_sentence'] = df.eval("sentence.str.contains('Nias')")
# Also assign sentence before and after to relevant_sentence (if it's from the same article (i.e. filename))
df['is_same_as_previous'] = df['filename'] == df['filename'].shift(1)
df['is_same_as_next'] = df['filename'] == df['filename'].shift(-1)

# Set relevant_sentence to True for sentences before and after relevant sentences
df['relevant_sentence'] = df['relevant_sentence'] | (df['is_same_as_previous'] & df['relevant_sentence'].shift(1, fill_value=False)) | (df['is_same_as_next'] & df['relevant_sentence'].shift(-1, fill_value=False))


df = df.query("relevant_sentence")
df

### Create dataframe with all persons

In [ ]:
# Create a dataframe with only persons, one row per person
persons = df[['filename', 'person']]\
    .explode('person')\
    .dropna(subset=['person'])\
    .reset_index(drop=True)

persons_df = pd.concat([persons, pd.DataFrame(persons['person'].tolist())], axis=1).drop(columns=['person'])
print(f"Unieke personen: {persons_df.text.nunique()}")
persons_df

## Convert to embeddings, build and search index

We're creating a 'FAISS' vector-index, to store the embeddings, and which we can use to quickly search and retrieve candidates (https://github.com/facebookresearch/faiss). Below is just a basic index, we could optimize it a bit if it turns out to be too slow

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

# https://rapidfuzz.github.io/RapidFuzz/index.html
from rapidfuzz import fuzz, distance

In [ ]:
# !git clone https://huggingface.co/emanjavacas/GysBERT

In [ ]:
# import torch
# from sentence_transformers import SentenceTransformer

# embed_model = "GysBERT"

# if torch.cuda.is_available():
#     device = "cuda"
# elif torch.backends.mps.is_available():
#     device = "mps"
# else:
#     device = "cpu"

# # 1. Load a pretrained Sentence Transformer model
# model = SentenceTransformer(embed_model, device=device, trust_remote_code=True)

In [ ]:
# !hf download emanjavacas/GysBERT

In [ ]:
import torch

# _original_torch_load = torch.load

# def torch_load_compat(*args, **kwargs):
#     kwargs.setdefault("weights_only", False)
#     return _original_torch_load(*args, **kwargs)

# torch.load = torch_load_compat

from sentence_transformers import SentenceTransformer

embed_model = "emanjavacas/GysBERT"

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer(embed_model, device=device)

In [ ]:
## Create embeddings for all names in the dataframe
# Get list of unique names
names = persons_df.text.unique().tolist()
print(f"Number of unique names: {len(names)}")

# make a dictionary of name to index 
name_to_index = {name: i for i, name in enumerate(names)}

# map the current index of the dataframe to the index of the name in the list of unique names
index_to_person_df_index = {i: persons_df.index[persons_df.text == name].tolist()[0] for i, name in enumerate(names)}

# Create embeddings for the unique names
embeddings = model.encode(names)


In [ ]:
# now based on the index_to_person_df_index we can map the index of the name in the list of unique names to the index of the dataframe

name_index_map_df = pd.DataFrame([
    {
        "unique_name_index": i,
        "person_df_index": index_to_person_df_index[i],
        "name": names[i],
        "person_df_name": persons_df.loc[index_to_person_df_index[i], "text"],
        "person_df_article": persons_df.loc[index_to_person_df_index[i], "filename"],
    }
    for i in range(len(names))
])

In [ ]:
name_index_map_df

In [ ]:
# Create index
index = faiss.IndexFlatL2(embeddings.shape[1])
print(index.is_trained)

# Add embeddings to the index
index.add(embeddings)

print(index.ntotal)

In [ ]:
# Get k nearest neighbors for each name
k = 10

DISTANCES, CANDIDATE_INDICES = index.search(embeddings, k)

In [ ]:
filepath = Path("DST/similar_names.json")

In [ ]:
# create a json object with names, article id and candidate with top 10 similar names along with thier similarity score and article id

# create a json object with names, article id and candidate with top 10 similar names along with thier similarity score and article id

json_object = [
    {
        "name": names[i],
        "article": persons_df.loc[index_to_person_df_index[i], "filename"],
        "similar_names": [
            {
                "name": names[j],
                "similarity": round(score, 2),
                "article": persons_df.loc[index_to_person_df_index[j], "filename"]
            }
            for j in CANDIDATE_INDICES[i, :]
            if (score := fuzz.WRatio(names[i], names[j])) > 50
        ]
    }
    for i in range(len(names))
]

# suggested_names = pd.DataFrame({
#     "name": names,
#     "similar_names": [
#         {
#             names[j]: round(score, 2)
#             for j in CANDIDATE_INDICES[i, :]
#             if (score := fuzz.WRatio(names[i], names[j])) > 50
#         }
#         for i in range(len(names))
#     ],
#     "article": [
#         persons_df.loc[index_to_person_df_index[i], "filename"]
#         for i in range(len(names))
#     ]
# })

import json
json.dump(json_object, open(filepath, "w+"), indent=4)

In [ ]:
# convert the json object to networkx graph

import json

def convert_to_graph(records):
    node_map = {}
    nodes = []
    edges = []

    # Keep stable IDs
    def add_node(name, label, article):
        if name not in node_map:
            node_id = len(node_map) + 1
            node_map[name] = node_id
            nodes.append({
                "data": {
                    "id": node_id,
                    "label": label,
                    "name": name,
                    "article": article
                }
            })
        return node_map[name]

    edge_id = 20

    for item in records:
        source_name = item.get("name")
        source_label = item.get("label", "PERSON")
        source_article = item.get("article")

        source_id = add_node(source_name, source_label, source_article)

        for related in item.get("similar_names", []):
            target_name = related.get("name")
            target_label = related.get("label", "PERSON") 
            target_article = related.get("article")

            target_id = add_node(target_name, target_label, target_article)

            edges.append({
                "data": {
                    "id": edge_id,
                    "label": related.get("relation", "SIMILAR"),
                    "score": related.get("similarity"),
                    "source": source_id,
                    "target": target_id
                }
            })
            edge_id += 1

    return {
        "nodes": nodes,
        "edges": edges
    }


# Example usage
with open(filepath, "r", encoding="utf-8") as f:
    data = json.load(f)

graph = convert_to_graph(data)

with open("DST/graph_output.json", "w+", encoding="utf-8") as f:
    json.dump(graph, f, ensure_ascii=False, indent=2)

### Inspect results

In [ ]:
# Name idx (from list of unique names) to inspect
name_idx = 1


# Levenshtein weights for insertion, deletion, substitution
levenshtein_weights = (2, 2, 1)  

name = names[name_idx]

# Create a DataFrame to display the results (one row per name-candidate pair)
results = pd.DataFrame({
    "name": [name] * k,
    "candidate": [names[j] for j in CANDIDATE_INDICES[name_idx]],
    "l2_distance": DISTANCES[name_idx]
})

## Add columns with fuzzy matching scores
results['wratio'] = results['candidate'].apply(lambda x: fuzz.WRatio(name, x))
results['token_sort_ratio'] = results['candidate'].apply(lambda x: fuzz.token_sort_ratio(name, x))
results['lcs_ratio'] = results['candidate'].apply(lambda x: distance.LCSseq.normalized_distance(name.lower(), x.lower()))
results['levenshtein_ratio'] = results['candidate'].apply(lambda x: distance.Levenshtein.normalized_distance(name.lower(), x.lower(), weights=levenshtein_weights))
results['jarowinkler'] = results['candidate'].apply(lambda x: distance.JaroWinkler.normalized_distance(name.lower(), x.lower()))

# Display results rounded to 2 decimal places for readability
results.round(2)

In [ ]:
# based on the embedding give me top 10 names that are similar to the name in the first column. Then for each of those names, give me the levenshtein distance, 


